In [10]:
from datetime import datetime, timedelta

"""
DENTAL CLINIC SMART QUEUE & LOAD BALANCER - FINAL VERSION
========================================================
This system implements a robust, object-oriented solution for patient queue management
and dentist workload balancing.

Key Design Principles:
- Separation of Concerns (Dentist manages state, Clinic manages queue/assignment logic).
- Priority Queueing (Appointments take precedence over walk-ins).
- Preference Handling (Specific dentist requests are honored only if free).
- Load Balancing Metric (Assignment prioritizes the dentist who is currently free).
"""



# GLOBAL CONSTANTS

DENTIST_NAMES = ["Dr. Ayesha", "Dr. Fatima", "Dr. Sana", "Dr. Hira"]
DENTAL_SPECIALTIES = ["General Dentistry", "Endodontist", "Orthodontist", "Periodontist"]


# 1. Dentist Class (The Core Resource)

class Dentist:
    """Represents a single dentist, tracking their name and availability status."""
    def __init__(self, name, specialty):
        self.name = name
        self.specialty = specialty
        self.busy_until = None
        self.current_patient = None

    def is_free(self):
        """Returns True if the dentist is currently free. If the scheduled time has passed,
        clear the busy status and return True."""
        if self.busy_until is None:
            return True

        # If the scheduled time has passed, clear status return true.
        if datetime.now() >= self.busy_until:
            self.busy_until = None
            self.current_patient = None
            return True

        return False

    def assign_patient(self, patient_name, duration_minutes):
        """Assigns a patient and makes the dentist busy for the required duration (minutes)."""
        # Ensure duration is integer minutes and at least 1
        duration_minutes = int(duration_minutes)
        if duration_minutes < 1:
            duration_minutes = 1

        start_time = datetime.now()
        # Calculate the future finish timestamp.
        self.busy_until = start_time + timedelta(minutes=duration_minutes)
        self.current_patient = patient_name

    def get_remaining_time(self):
        """Returns remaining minutes until dentist is free (0 if free)."""
        if self.busy_until is None:
            return 0

        remaining_seconds = (self.busy_until - datetime.now()).total_seconds()
        if remaining_seconds <= 0:
            return 0
        # Return remaining minutes rounded up to the next whole minute
        return int((remaining_seconds + 59) // 60)


In [11]:



# 2. Dental Clinic Queue System Class (The Controller)

class DentalClinicQueueSystem:
    """Manages all dentists, the patient queue, and the load balancing logic."""
    def __init__(self):
        # Initialize the 4 dentists
        self.dentists = [Dentist(DENTIST_NAMES[i], DENTAL_SPECIALTIES[i]) for i in range(4)]
        self.queue = []
        self.total_patients_processed = 0
        self.total_wait_time = 0  # In minutes

    # Helper: find dentist object by exact name
    def _get_dentist_by_name(self, name):
        for d in self.dentists:
            if d.name == name:
                return d
        return None

    def register_patient(self, patient_data):
        """
        Registers a patient using a simple dict input.
        Basic validation and priority insertion (appointments before walk-ins).
        """
        required_keys = ['name', 'treatment', 'duration', 'is_appointment']
        if not all(k in patient_data for k in required_keys):
            return {"status": "error", "message": "Missing required patient data keys."}

        name = patient_data['name']
        duration = patient_data['duration']
        preferred_dentist = patient_data.get('preferred', None)

        if not isinstance(name, str) or not name:
            return {"status": "error", "message": "Invalid patient name."}
        if not isinstance(duration, (int, float)) or duration <= 0:
            return {"status": "error", "message": "Duration must be a positive number."}

        # normalize preferred None (user might pass string 'None')
        if preferred_dentist == 'None':
            preferred_dentist = None

        if preferred_dentist and preferred_dentist not in DENTIST_NAMES:
            return {"status": "error", "message": f"Preferred dentist '{preferred_dentist}' not found."}

        patient = {
            "name": name,
            "treatment": patient_data['treatment'],
            "duration": int(duration),
            "preferred": preferred_dentist,
            "is_appointment": patient_data['is_appointment'],
            "time_joined": datetime.now()  # timestamp for wait-time metric
        }

        # Priority Logic: Appointments go before walk-ins (FIFO inside each group)
        if patient['is_appointment']:
            last_appt_index = -1
            for i, p in enumerate(self.queue):
                if p['is_appointment']:
                    last_appt_index = i
                else:
                    break
            self.queue.insert(last_appt_index + 1, patient)
            return {"status": "success", "message": f"Priority Appointment Registered: {name}"}
        else:
            self.queue.append(patient)
            return {"status": "success", "message": f"Walk-in Registered: {name}"}

    def find_least_busy_dentist(self):
        """Finds the dentist who will be available soonest (minimum remaining time)."""
        least_busy_time = float('inf')
        chosen_dentist = None

        for d in self.dentists:
            remaining_time = d.get_remaining_time()
            if remaining_time < least_busy_time:
                least_busy_time = remaining_time
                chosen_dentist = d

        return chosen_dentist

    def assign_patient_from_queue(self):
        """
        [Assignment Executor] Executes the core business logic using a decision tree:
        1. Handle Preferred Dentist (Strict Assignment Rule).
        2. Handle General Patient (Load Balance Assignment Rule).
        """
        if not self.queue:
            return {"status": "info", "message": "Queue is empty."}

        patient_to_assign = self.queue[0]  # highest priority
        assigned_dentist = None

        preferred_name = patient_to_assign.get("preferred")

        # If patient requested a preferred dentist:
        if preferred_name:
            preferred = self._get_dentist_by_name(preferred_name)
            # Patient waits if the preferred doctor is busy.
            if preferred is None:
                return {"status": "error", "message": f"Preferred dentist '{preferred_name}' not found."}
            # If preferred dentist is free now assign
            if preferred.is_free():
                assigned_dentist = preferred
            else:
                return {
                    "status": "info",
                    "message": f"Preferred dentist ({preferred_name}) is currently busy. Patient {patient_to_assign['name']} will wait."
                }
        else:
            # Assign to the first available (free) dentist found.
            for d in self.dentists:
                if d.is_free():
                    assigned_dentist = d
                    break

            # If none free right now, do not assign (we could schedule in future, but current logic waits)
            if assigned_dentist is None:
                return {
                    "status": "info",
                    "message": f"Highest priority patient ({patient_to_assign['name']}) is waiting. All dentists are busy."
                }

        # EXECUTE assignment
        if assigned_dentist:
            assigned_dentist.assign_patient(patient_to_assign["name"], patient_to_assign["duration"])
            self.track_metrics(patient_to_assign)
            self.queue.pop(0)
            return {
                "status": "success",
                "message": f"Patient {patient_to_assign['name']} successfully assigned to {assigned_dentist.name}."
            }

        # Fallback (shouldn't reach here)
        return {"status": "info", "message": "Unable to assign patient at this time."}

    def track_metrics(self, patient):
        """Calculates and updates performance metrics (Total Processed, Wait Time)."""
        wait_time = (datetime.now() - patient["time_joined"]).total_seconds() / 60
        self.total_wait_time += wait_time
        self.total_patients_processed += 1

    #  STATUS & TIME TRACKING
    def show_status(self):
        """Displays real-time status of all dentists and the queue."""
        print("\n" + "="*50)
        print(" CLINIC STATUS & LOAD BALANCING METRICS ".center(50))
        print("="*50)

        # Dentist Status
        print("\n--- DENTIST AVAILABILITY ---")
        for d in self.dentists:
            remaining_time = d.get_remaining_time()
            if remaining_time == 0:
                print(f"  [FREE] {d.name} ({d.specialty})")
            else:
                print(f"  [BUSY] {d.name} | Patient: {d.current_patient} | Finish in {remaining_time} min")

        # Queue Status
        print("\n--- WAITING QUEUE (Priority Order) ---")
        if not self.queue:
            print("  No patients currently waiting.")
        else:
            for i, p in enumerate(self.queue):
                priority = "APPT" if p["is_appointment"] else "WALK-IN"
                pref = f" (Pref: {p['preferred']})" if p['preferred'] else ""
                print(f"  {i+1}. [{priority}] {p['name']} ({p['duration']} min){pref}")

    def show_metrics(self):
        """Display performance metrics."""
        print("\n" + "-"*50)
        print(" PERFORMANCE METRICS (Success Reporter) ".center(50))
        print("-"*50)

        if self.total_patients_processed > 0:
            avg_wait = self.total_wait_time / self.total_patients_processed
            print(f" Total Patients Processed: {self.total_patients_processed}")
            print(f" Average Wait Time (AWT): {avg_wait:.2f} minutes")
        else:
            print(" No patients processed yet.")



In [ ]:

# 3. Interactive CLI Application


def register_patient_cli(clinic):
    """Gathers input from the user via CLI and registers the patient."""
    print("\n--- REGISTER NEW PATIENT ---")
    name = input("Patient Name: ").strip()
    treatment = input("Treatment Type: ").strip()

    # Simple input loop for duration
    while True:
        try:
            duration = int(input("Duration (minutes): ").strip())
            if duration < 1:
                print("ERROR: Must be at least 1 minute.")
                continue
            break
        except ValueError:
            print("ERROR: Invalid number.")

    # Priority selection
    priority_choice = input("Priority (A=Appointment, W=Walk-in): ").upper().strip()
    is_appointment = priority_choice == 'A'

    # Preferred Dentist selection
    print(f"Available dentists: {', '.join(DENTIST_NAMES)}")
    preferred = input("Preferred Dentist (Type name or 'None'): ").strip()
    preferred = preferred if preferred and preferred != 'None' else None

    patient_data = {
        'name': name,
        'treatment': treatment,
        'duration': duration,
        'preferred': preferred,
        'is_appointment': is_appointment
    }

    result = clinic.register_patient(patient_data)
    if result['status'] == 'success':
        print(f"SUCCESS: {result['message']}")
    else:
        print(f"ERROR: {result['message']}")

def assign_patient_cli(clinic):
    """Runs the core assignment logic once (assign highest priority patient if possible)."""
    print("\n--- RUNNING LOAD BALANCER ---")
    result = clinic.assign_patient_from_queue()
    if result['status'] == 'success':
        print(f"SUCCESS: {result['message']}")
    elif result['status'] == 'info':
        print(f"Info: {result['message']}")
    else:
        print(f"Error: {result['message']}")

def main():
    clinic = DentalClinicQueueSystem()
    print("="*50)
    print("Welcome to the Smart Queue Management System!")
    print("="*50)

    while True:
        print("\nAvailable commands: register, assign, status, metrics, quit")
        action = input("Action: ").lower().strip()

        if action == 'register':
            register_patient_cli(clinic)
        elif action == 'assign':
            assign_patient_cli(clinic)
        elif action == 'status':
            clinic.show_status()
        elif action == 'metrics':
            clinic.show_metrics()
        elif action == 'quit':
            print("\nSystem shutting down. Goodbye!")
            break
        else:
            print("ERROR: Invalid command.")

    return clinic

if __name__ == "__main__":
    # Run interactive loop
    main()


Welcome to the Smart Queue Management System!

Available commands: register, assign, status, metrics, quit

--- REGISTER NEW PATIENT ---
Available dentists: Dr. Ayesha, Dr. Fatima, Dr. Sana, Dr. Hira
SUCCESS: Walk-in Registered: Humaira

Available commands: register, assign, status, metrics, quit

--- RUNNING LOAD BALANCER ---
SUCCESS: Patient Humaira successfully assigned to Dr. Ayesha.

Available commands: register, assign, status, metrics, quit

      CLINIC STATUS & LOAD BALANCING METRICS      

--- DENTIST AVAILABILITY ---
  [BUSY] Dr. Ayesha | Patient: Humaira | Finish in 14 min
  [FREE] Dr. Fatima (Endodontist)
  [FREE] Dr. Sana (Orthodontist)
  [FREE] Dr. Hira (Periodontist)

--- WAITING QUEUE (Priority Order) ---
  No patients currently waiting.

Available commands: register, assign, status, metrics, quit
